In [5]:
!pip install dash==2.9.3 jupyter-dash==0.4.2 dash-bootstrap-components==1.4.1 plotly --quiet

In [22]:
# ==============================
# WAR INTELLIGENCE DASHBOARD
# ==============================

import pandas as pd
import numpy as np
import plotly.express as px

from jupyter_dash import JupyterDash
from dash import dcc, html, Input, Output
import dash_bootstrap_components as dbc

# ==============================
# LOAD DATA
# ==============================
df = pd.read_csv("war_news_events_dataset.csv")

df.columns = df.columns.str.strip()

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date'])

df['Fatalities'] = pd.to_numeric(df['Fatalities'], errors='coerce').fillna(0)

df['Month'] = df['Date'].dt.to_period('M').astype(str)

# ==============================
# FILTER VALUES
# ==============================
all_conflicts = sorted(df['Conflict'].dropna().unique())
all_event_types = sorted(df['Event_Type'].dropna().unique())
all_countries = sorted(df['Country'].dropna().unique())
all_sources = sorted(df['Source'].dropna().unique())

MIN_DATE = df['Date'].min()
MAX_DATE = df['Date'].max()

# ==============================
# FILTER FUNCTION
# ==============================
def apply_filters(start, end, c, e, co, s):
    fdf = df.copy()

    if start:
        fdf = fdf[fdf['Date'] >= pd.to_datetime(start)]
    if end:
        fdf = fdf[fdf['Date'] <= pd.to_datetime(end)]

    if c and 'ALL' not in c:
        fdf = fdf[fdf['Conflict'].isin(c)]
    if e and 'ALL' not in e:
        fdf = fdf[fdf['Event_Type'].isin(e)]
    if co and 'ALL' not in co:
        fdf = fdf[fdf['Country'].isin(co)]
    if s and 'ALL' not in s:
        fdf = fdf[fdf['Source'].isin(s)]

    return fdf

# ==============================
# APP INIT
# ==============================
app = JupyterDash(__name__, external_stylesheets=[dbc.themes.CYBORG])

# ==============================
# LAYOUT
# ==============================
app.layout = dbc.Container([

    html.Div([
        html.H2("🌍 War Intelligence Dashboard", style={"color": "#c9d1d9"}),
        html.P("Interactive analysis of global conflict patterns",
               style={"color": "#8b949e"})
    ], style={"marginBottom": "20px"}),

    dbc.Row([

        # SIDEBAR
        dbc.Col([

            html.H5("Filters", style={"color": "#58a6ff"}),

            dcc.DatePickerRange(
                id='date-range',
                start_date=MIN_DATE,
                end_date=MAX_DATE
            ),

            dcc.Dropdown(
                options=[{'label': 'Select All', 'value': 'ALL'}] +
                        [{'label': i, 'value': i} for i in all_conflicts],
                value=['ALL'],
                id='conflict-filter',
                multi=True
            ),

            dcc.Dropdown(
                options=[{'label': 'Select All', 'value': 'ALL'}] +
                        [{'label': i, 'value': i} for i in all_event_types],
                value=['ALL'],
                id='event-filter',
                multi=True
            ),

            dcc.Dropdown(
                options=[{'label': 'Select All', 'value': 'ALL'}] +
                        [{'label': i, 'value': i} for i in all_countries],
                value=['ALL'],
                id='country-filter',
                multi=True
            ),

            dcc.Dropdown(
                options=[{'label': 'Select All', 'value': 'ALL'}] +
                        [{'label': i, 'value': i} for i in all_sources],
                value=['ALL'],
                id='source-filter',
                multi=True
            ),

            html.Hr(),

            html.Div(id='kpis'),

            html.Hr(),

            html.H5("📊 Insights", style={"color": "#58a6ff"}),
            html.Div(id='insights')

        ], width=3, style={"background": "#161b22", "padding": "15px"}),

        # MAIN
        dbc.Col([

            dcc.Graph(id='map'),
            dcc.Graph(id='timeline'),

            dbc.Row([
                dbc.Col(dcc.Graph(id='comparison'), width=6),
                dbc.Col(dcc.Graph(id='event-type'), width=6)
            ]),

            # ✅ COMBINED BOX + SCATTER HERE
            dbc.Row([
                dbc.Col(dcc.Graph(id='fatality'), width=12)
            ]),

            dbc.Row([
                dbc.Col(dcc.Graph(id='source'), width=12)
            ])

        ], width=9)

    ])

], fluid=True, style={"backgroundColor": "#0d1117"})

# ==============================
# CALLBACKS
# ==============================

@app.callback(
    Output('kpis','children'),
    Input('date-range','start_date'),
    Input('date-range','end_date'),
    Input('conflict-filter','value'),
    Input('event-filter','value'),
    Input('country-filter','value'),
    Input('source-filter','value')
)
def kpis(start,end,c,e,co,s):
    fdf = apply_filters(start,end,c,e,co,s)

    return [
        html.H4(f"Events: {len(fdf)}"),
        html.H4(f"Fatalities: {int(fdf['Fatalities'].sum())}"),
        html.H4(f"Avg/Event: {round(fdf['Fatalities'].mean(),2)}")
    ]


@app.callback(
    Output('insights','children'),
    Input('date-range','start_date'),
    Input('date-range','end_date'),
    Input('conflict-filter','value'),
    Input('event-filter','value'),
    Input('country-filter','value'),
    Input('source-filter','value')
)
def insights(start,end,c,e,co,s):
    fdf = apply_filters(start,end,c,e,co,s)

    if len(fdf)==0:
        return "No data"

    return [
        html.P(f"🔥 Most active: {fdf['Conflict'].value_counts().idxmax()}"),
        html.P(f"💀 Deadliest type: {fdf.groupby('Event_Type')['Fatalities'].sum().idxmax()}"),
        html.P(f"📊 Avg fatalities: {round(fdf['Fatalities'].mean(),2)}")
    ]


# ==============================
# MAP
# ==============================
@app.callback(
    Output('map','figure'),
    Input('date-range','start_date'),
    Input('date-range','end_date'),
    Input('conflict-filter','value'),
    Input('event-filter','value'),
    Input('country-filter','value'),
    Input('source-filter','value')
)
def map_cb(start,end,c,e,co,s):
    fdf = apply_filters(start,end,c,e,co,s)

    fig = px.scatter_geo(
        fdf,
        lat='Latitude',
        lon='Longitude',
        color='Conflict',
        size='Fatalities',
        hover_name='Country',
        projection="natural earth"
    )

    fig.update_geos(
        showcountries=True,
        countrycolor="white",
        showcoastlines=True,
        coastlinecolor="gray",
        showland=True,
        landcolor="#1f2c3c"
    )

    fig.update_layout(template="plotly_dark", title="Global Conflict Map")

    return fig


# ==============================
# TIMELINE
# ==============================
@app.callback(
    Output('timeline','figure'),
    Input('date-range','start_date'),
    Input('date-range','end_date'),
    Input('conflict-filter','value'),
    Input('event-filter','value'),
    Input('country-filter','value'),
    Input('source-filter','value')
)
def timeline(start,end,c,e,co,s):
    fdf = apply_filters(start,end,c,e,co,s)

    g = fdf.groupby(['Month','Conflict']).size().reset_index(name='Events')

    fig = px.line(g,x='Month',y='Events',color='Conflict',markers=True)
    fig.update_layout(template="plotly_dark",title="Timeline")

    return fig


# ==============================
# COMPARISON (FATALITIES)
# ==============================
@app.callback(
    Output('comparison','figure'),
    Input('date-range','start_date'),
    Input('date-range','end_date'),
    Input('conflict-filter','value'),
    Input('event-filter','value'),
    Input('country-filter','value'),
    Input('source-filter','value')
)
def comp(start,end,c,e,co,s):
    fdf = apply_filters(start,end,c,e,co,s)

    g = fdf.groupby('Conflict')['Fatalities'].sum().reset_index()

    fig = px.bar(
        g,
        x='Conflict',
        y='Fatalities',
        color='Conflict',
        text='Fatalities'
    )

    fig.update_layout(template="plotly_dark",title="Total Fatalities per Conflict")

    return fig


# ==============================
# EVENT TYPE
# ==============================
@app.callback(
    Output('event-type','figure'),
    Input('date-range','start_date'),
    Input('date-range','end_date'),
    Input('conflict-filter','value'),
    Input('event-filter','value'),
    Input('country-filter','value'),
    Input('source-filter','value')
)
def event(start,end,c,e,co,s):
    fdf = apply_filters(start,end,c,e,co,s)

    g = fdf.groupby('Event_Type').size().reset_index(name='Count')

    fig = px.pie(g,names='Event_Type',values='Count')
    fig.update_layout(template="plotly_dark",title="Event Types")

    return fig


# ==============================
# COMBINED BOX + SCATTER
# ==============================
@app.callback(
    Output('fatality','figure'),
    Input('date-range','start_date'),
    Input('date-range','end_date'),
    Input('conflict-filter','value'),
    Input('event-filter','value'),
    Input('country-filter','value'),
    Input('source-filter','value')
)
def fatal(start,end,c,e,co,s):
    fdf = apply_filters(start,end,c,e,co,s)

    fig = px.box(
        fdf,
        x='Conflict',
        y='Fatalities',
        color='Conflict',
        points='all'
    )

    fig.update_traces(
        jitter=0.4,
        marker_size=5,
        opacity=0.6
    )

    fig.update_layout(
        template="plotly_dark",
        title="Fatalities Distribution (Box + Scatter)"
    )

    return fig


# ==============================
# HEATMAP
# ==============================
@app.callback(
    Output('source','figure'),
    Input('date-range','start_date'),
    Input('date-range','end_date'),
    Input('conflict-filter','value'),
    Input('event-filter','value'),
    Input('country-filter','value'),
    Input('source-filter','value')
)
def heat(start,end,c,e,co,s):
    fdf = apply_filters(start,end,c,e,co,s)

    pivot = fdf.pivot_table(index='Source',columns='Conflict',
                            values='Event_Type',aggfunc='count',fill_value=0)

    fig = px.imshow(pivot,text_auto=True)
    fig.update_layout(template="plotly_dark",title="Source Heatmap")

    return fig


# ==============================
# RUN
# ==============================
app.run_server(debug=True)

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



Dash app running on:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>